# Auditoría exploratoria del catálogo wellness de SMARTUR

Este notebook describe los archivos locales de destinos wellness y detecta campos faltantes, duplicados y rangos. **No entrena un clasificador, no mide estrés y no estima la eficacia de las recomendaciones.** El archivo estático de destinos no equivale al catálogo actual aprobado en PostgreSQL.

Ejecutar desde la raíz del repositorio, `MODELO/` o `MODELO/notebooks/`. Los resultados solo describen el archivo local cuya ruta se muestra.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [
    Path('data/wellness/destinos_wellness_enriched.csv'),
    Path('MODELO/data/wellness/destinos_wellness_enriched.csv'),
    Path('../data/wellness/destinos_wellness_enriched.csv'),
]
dataset_path = next((p.resolve() for p in candidates if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError('No se encontró data/wellness/destinos_wellness_enriched.csv')
df = pd.read_csv(dataset_path)
print(f'Archivo auditado: {dataset_path}')
print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')
display(df.head())

In [ ]:
# Perfilado descriptivo: no corrige ni imputa datos.
profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(dropna=True),
}).sort_values(['missing_pct', 'unique'], ascending=[False, True])
display(profile)

id_col = next((c for c in ['id_destino', 'id', 'place_id'] if c in df.columns), None)
print('Columna candidata de identificador:', id_col)
if id_col:
    print('Filas con identificador duplicado:', int(df[id_col].duplicated().sum()))
else:
    print('No hay una columna de identificador reconocida; no se calcula duplicidad.')

In [ ]:
# Distribuciones descriptivas de categorías y campos numéricos presentes.
for col in ['estado', 'categoria_wellness', 'categoria_principal', 'fuente']:
    if col in df.columns:
        print(f'\n{col}:')
        display(df[col].fillna('(sin dato)').value_counts().head(20).rename('registros').to_frame())

numeric_cols = [c for c in ['nivel_aislamiento', 'restauracion_pasiva', 'demanda_fisica', 'wellness_sentiment_score'] if c in df.columns]
if numeric_cols:
    display(df[numeric_cols].describe().T)
    df[numeric_cols].hist(bins=12, figsize=(12, 6), layout=((len(numeric_cols) + 1) // 2, 2))
    plt.tight_layout()
    plt.show()
else:
    print('El dataset no contiene atributos numéricos de bienestar reconocidos.')

In [ ]:
# Auditoría de aptitud para entrenar/validar una recomendación personalizada.
required_for_personalized_validation = {'user_id', 'item_id', 'wellness_dimensions', 'relevance_label'}
available = set(df.columns)
missing_for_validation = sorted(required_for_personalized_validation - available)
print('Campos requeridos para evaluación personalizada que faltan:', missing_for_validation)
print('¿Permite calcular métricas de relevancia personalizadas?', not missing_for_validation)
print('¿Permite entrenar detección de estrés?', False, '(no se aceptan etiquetas clínicas inferidas del nombre, reseñas o categoría)')
print('Conclusión: usar este archivo para inspección de catálogo y pruebas técnicas; no para afirmar precisión, satisfacción o efecto terapéutico.')

## Lectura de resultados

La ausencia de IDs de usuario, elecciones explícitas y etiquetas de relevancia impide calcular Precision@K, Recall@K o nDCG@K con significado. Para esas métricas se necesita un conjunto de casos de consulta y juicios de relevancia registrados mediante un protocolo reproducible. Para una evaluación de estrés se requerirían datos recogidos con instrumento autorizado, protocolo ético y etiquetas válidas; este archivo no cumple esa función. Los valores sintéticos se reservan para pruebas del comportamiento del código y no se reportan como rendimiento del modelo.